# Phase C — Diffusion Policy on Kaggle

This notebook trains the official LeRobot Diffusion Policy on the private, sanitized Phase B dataset. It does not upload the trained policy.

Before running:

1. Select GPU T4 x2 and turn Internet on in Kaggle settings.
2. Add a Kaggle secret named HF_TOKEN with read access to stevenzenith/hand_tracking_pv_carton_phase_b.
3. Use Save Version → Save & Run All for the full run so /kaggle/working is retained.

Frozen first-run contract: 30 episodes / 6640 frames / 10 Hz; front and side RGB plus six-joint state; native two-step observation window [-1, 0]; 16-step prediction horizon and 8 executed actions; no PV, teacher, load, temperature, grip-context, current target action, or Phase A data.

The 90k run keeps 20k/40k/60k/80k/90k checkpoints. With the initial global batch 4 and seven dropped tail frames per episode, these are approximately 12.4/24.9/37.3/49.8/56.0 sampler epochs. Select a checkpoint by held-out or robot rollout performance, not by the lowest training loss alone.

In [ ]:
# Install the same LeRobot source revision used by ACT and SmolVLA.
import importlib.metadata as metadata
import subprocess
import sys

LEROBOT_COMMIT = "da92db8fc0c935950a56b1ea61fa9b211ef3ac30"
assert sys.version_info >= (3, 12), (
    f"This pinned LeRobot revision requires Python >=3.12; Kaggle has {sys.version.split()[0]}"
)

package = f"lerobot[training,diffusion] @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        package,
        "huggingface_hub==1.19.0",
        "transformers==5.5.4",
    ],
    check=True,
)

for name in ["lerobot", "torch", "torchvision", "torchcodec", "diffusers", "accelerate", "huggingface_hub"]:
    print(f"{name}={metadata.version(name)}")

In [ ]:
# Authenticate, verify the requested accelerator, and pin/download the private dataset.
import os
from pathlib import Path

SCRATCH_ROOT = Path("/kaggle/temp/phase_c_diffusion")
WORK_ROOT = Path("/kaggle/working/phase_c_diffusion")
DATASET_ROOT = SCRATCH_ROOT / "dataset"
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(SCRATCH_ROOT / "hf_home")
os.environ["HF_DATASETS_CACHE"] = str(SCRATCH_ROOT / "hf_home" / "datasets")
os.environ["TORCH_HOME"] = str(SCRATCH_ROOT / "torch_home")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download
import torch

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_memory_gib = [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2) for i in range(torch.cuda.device_count())]
print("GPUs:", list(zip(gpu_names, gpu_memory_gib, strict=True)))
print("CUDA:", torch.version.cuda)
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    "Select the Kaggle GPU T4 x2 accelerator before continuing"
)

DATASET_ID = "stevenzenith/hand_tracking_pv_carton_phase_b"
DATASET_REVISION = "1bb681ab58b5ca2cbdedb52dabf8e1f7a6052a6a"
dataset_info = api.dataset_info(DATASET_ID, revision=DATASET_REVISION)
assert dataset_info.private is True
assert dataset_info.sha == DATASET_REVISION
print("dataset revision:", dataset_info.sha, "private:", dataset_info.private)

snapshot_download(
    repo_id=DATASET_ID, repo_type="dataset", revision=DATASET_REVISION,
    local_dir=DATASET_ROOT, token=hf_token,
)
del hf_token

In [ ]:
# Verify the dataset and the exact Diffusion temporal contract before allocating the model.
import math
from lerobot.configs import FeatureType
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion import DiffusionConfig
from lerobot.utils.feature_utils import dataset_to_policy_features
from torchvision.models import ResNet18_Weights

dataset = LeRobotDataset(
    repo_id=DATASET_ID, root=DATASET_ROOT, revision=DATASET_REVISION,
    video_backend="torchcodec",
)
assert dataset.num_episodes == 30
assert dataset.num_frames == 6640
assert dataset.fps == 10
assert set(dataset.meta.camera_keys) == {"observation.images.front", "observation.images.side"}

sample = dataset[0]
assert tuple(sample["observation.images.front"].shape) == (3, 480, 640)
assert tuple(sample["observation.images.side"].shape) == (3, 480, 640)
assert tuple(sample["observation.state"].shape) == (6,)
assert tuple(sample["action"].shape) == (6,)

features = dataset_to_policy_features(dataset.meta.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}
expected_inputs = {"observation.images.front", "observation.images.side", "observation.state"}
assert set(input_features) == expected_inputs
assert set(output_features) == {"action"}

contract = DiffusionConfig(
    input_features=input_features, output_features=output_features,
    n_obs_steps=2, horizon=16, n_action_steps=8,
)
assert contract.observation_delta_indices == [-1, 0]
assert contract.action_delta_indices == list(range(-1, 15))
assert contract.drop_n_last_frames == 7

# Fetch the built-in pretrained ResNet weights once before the two DDP workers start.
ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=True)
print("dataset:", dataset.num_episodes, "episodes /", dataset.num_frames, "frames /", dataset.fps, "Hz")
print("sample shapes:", {key: tuple(sample[key].shape) for key in sorted(expected_inputs | {"action"})})
print("observation indices:", contract.observation_delta_indices, "action indices:", contract.action_delta_indices)
del dataset, sample

In [ ]:
# Build the fixed two-GPU command and stream a persistent log without shell interpolation.
import datetime as dt
import json
import shutil
import subprocess

TRAIN_CLI = shutil.which("lerobot-train")
ACCELERATE = shutil.which("accelerate")
assert TRAIN_CLI and ACCELERATE
PER_GPU_BATCH = 2
NUM_GPUS = 2
GLOBAL_BATCH = PER_GPU_BATCH * NUM_GPUS
VALID_FRAMES = 6640 - 30 * 7
STEPS_PER_EPOCH = math.ceil(VALID_FRAMES / GLOBAL_BATCH)
print("global batch:", GLOBAL_BATCH, "valid sampler frames:", VALID_FRAMES, "steps/epoch:", STEPS_PER_EPOCH)

def build_command(*, steps: int, output_dir: Path, job_name: str, save_checkpoint: bool, save_freq: int, log_freq: int) -> list[str]:
    warmup_steps = 500 if steps >= 90_000 else 20
    train_args = [
        "--policy.type=diffusion",
        f"--dataset.repo_id={DATASET_ID}",
        f"--dataset.root={DATASET_ROOT}",
        f"--dataset.revision={DATASET_REVISION}",
        "--dataset.video_backend=torchcodec",
        f"--batch_size={PER_GPU_BATCH}",
        "--num_workers=2",
        "--policy.n_obs_steps=2",
        "--policy.horizon=16",
        "--policy.n_action_steps=8",
        "--policy.drop_n_last_frames=7",
        "--policy.device=cuda",
        "--policy.use_amp=true",
        "--policy.push_to_hub=false",
        f"--policy.scheduler_warmup_steps={warmup_steps}",
        f"--steps={steps}",
        "--eval_freq=0",
        f"--save_checkpoint={str(save_checkpoint).lower()}",
        f"--save_freq={save_freq}",
        f"--log_freq={log_freq}",
        f"--output_dir={output_dir}",
        f"--job_name={job_name}",
        "--seed=1000",
        "--wandb.enable=false",
    ]
    return [
        ACCELERATE, "launch", "--multi_gpu", "--num_processes=2",
        "--mixed_precision=fp16", "--main_process_port=29518",
        TRAIN_CLI, *train_args,
    ]

def run_and_log(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    child_env["TQDM_DISABLE"] = "1"
    print("command:", subprocess.list2cmdline(command))
    with log_path.open("w", buffering=1) as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=child_env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training exited with code {return_code}; inspect {log_path}")

In [ ]:
# Mandatory 20-update distributed smoke: AV1 history decode, forward/backward, fp16 and memory.
import re
import statistics

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
SMOKE_OUTPUT = WORK_ROOT / "outputs" / f"smoke_{stamp}"
SMOKE_LOG = WORK_ROOT / "logs" / f"smoke_{stamp}.log"
smoke_command = build_command(
    steps=20, output_dir=SMOKE_OUTPUT, job_name="diffusion_phase_c_smoke",
    save_checkpoint=False, save_freq=20, log_freq=1,
)
run_and_log(smoke_command, SMOKE_LOG)

smoke_text = SMOKE_LOG.read_text(errors="replace")
timings = [(float(update), float(data)) for update, data in re.findall(r"updt_s:([0-9.]+).*?data_s:([0-9.]+)", smoke_text)]
memory = [float(value) for value in re.findall(r"mem_gb:([0-9.]+)", smoke_text)]
losses = [float(value) for value in re.findall(r"loss:([0-9.]+)", smoke_text)]
assert timings and memory and losses, "Smoke completed without parseable finite metrics"
steady = timings[-min(5, len(timings)):]
seconds_per_step = statistics.mean(update + data for update, data in steady)
estimated_hours = seconds_per_step * 90_000 / 3600
print(f"smoke passed: {seconds_per_step:.3f} s/step steady, peak logged memory {max(memory):.2f} GiB")
print(f"projected 90k update time: {estimated_hours:.2f} h, excluding setup and checkpoints")

In [ ]:
# Full first baseline. This intentionally starts fresh instead of continuing the smoke weights.
FULL_STEPS = 90_000
full_stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
FULL_OUTPUT = WORK_ROOT / "outputs" / f"diffusion_phase_c_90k_{full_stamp}"
FULL_LOG = WORK_ROOT / "logs" / f"diffusion_phase_c_90k_{full_stamp}.log"
full_command = build_command(
    steps=FULL_STEPS, output_dir=FULL_OUTPUT, job_name="diffusion_phase_c_90k",
    save_checkpoint=True, save_freq=20_000, log_freq=50,
)
run_and_log(full_command, FULL_LOG)

manifest = {
    "completed_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "lerobot_commit": LEROBOT_COMMIT,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "gpus": gpu_names,
    "per_gpu_batch": PER_GPU_BATCH,
    "global_batch": GLOBAL_BATCH,
    "valid_sampler_frames": VALID_FRAMES,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "steps": FULL_STEPS,
    "approx_epochs": FULL_STEPS / STEPS_PER_EPOCH,
    "n_obs_steps": 2,
    "horizon": 16,
    "n_action_steps": 8,
    "drop_n_last_frames": 7,
    "mixed_precision": "fp16",
    "smoke_seconds_per_step": seconds_per_step,
    "smoke_projected_90k_hours": estimated_hours,
    "output_dir": str(FULL_OUTPUT),
    "log_path": str(FULL_LOG),
    "command": full_command,
}
manifest_path = WORK_ROOT / "diffusion_phase_c_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("manifest:", manifest_path)

In [ ]:
# Confirm retained checkpoints before saving the Kaggle notebook version.
checkpoint_models = sorted(
    model_path for model_path in (FULL_OUTPUT / "checkpoints").glob("*/pretrained_model/model.safetensors")
    if model_path.parents[1].name.isdigit()
)
assert len(checkpoint_models) == 5, f"Expected 20k/40k/60k/80k/90k checkpoints, found {len(checkpoint_models)}"
for model_path in checkpoint_models:
    print(model_path.relative_to(WORK_ROOT), f"{model_path.stat().st_size / 2**20:.1f} MiB")
print("full log:", FULL_LOG)
print("Now save this Kaggle notebook version so /kaggle/working/phase_c_diffusion is retained.")